Anfangs war ich gegen RoBERTa, da mein Laptop keine GPU hat und ich mich noch nicht mit Google Collab auseinandergesetzt hatte. Nun wollte ich mal testen, ob das RoBERTa als leistungsfähigstes Modell die Einwürfe sinnvoll vorhersagen kann. In diesem Notebook werde ich zeigen, wie ich vorgegangen bin und was am Ende raus kam.

Anmerkung: RoBERTa Modell ist sehr rechenintensiv, weshalb nur auf 10000 Einträgen trainiert wurde.

In [ ]:
! pip install datasets
! pip install torch
! pip install evaluate
!pip install optuna transformers torch

! pip install transformers==4.28.1
! pip install accelerate==0.15.0
! pip install tokenizers==0.13.3
!pip install --upgrade google-auth-oauthlib google-auth-httplib2 google-api-python-client google-auth

In [ ]:
import json
import torch
from transformers import XLMRobertaTokenizer, XLMRobertaForTokenClassification, Trainer, TrainingArguments, XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from torch.utils.data import Dataset, random_split
import torch.nn as nn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
file_path = '/content/drive/My Drive/Colab_Notebooks/modified_texts_with_interruption.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

data = data[:10000]

texts = []
labels = []

for entry in data:
    text = entry["modified_text"]
    texts.append(text)

    label = []
    tokens = text.split()
    for token in tokens:
        if "<interruption>" in token:
            label.append(1)
        else:
            label.append(0)
    labels.append(label)

max_len = 512
padded_labels = [label + [0] * (max_len - len(label)) if len(label) < max_len else label[:max_len] for label in labels]

example_texts = texts[:5]
example_labels = labels[:5]

for i, text in enumerate(example_texts):
    print(f"Text {i+1}: {text}")
    print(f"Labels {i+1}: {example_labels[i]}")

for i, label in enumerate(padded_labels[:5]):
    print(f"Padded Labels {i+1}: {label}")

class InterruptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        labels = torch.tensor(labels[:self.max_len] + [0] * (self.max_len - len(labels)), dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

Text 1: Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Verehrte Bürger! In diesen Coronazeiten – wir haben es in dieser Woche schon öfter gehört – ist wenig normal. Die Bewältigung der Coronakrise hat erhebliche Auswirkungen auf den Bundeshaushalt.
Besonders davon betroffen ist – kein Wunder – der Bereich des Einzelplans 11, Arbeit und Soziales. Der Haushaltsentwurf 2021 für das Bundesministerium für Arbeit und Soziales hat einen Umfang von insgesamt rund 165 Milliarden Euro und ist damit wieder der größte Einzelplan im Bundeshaushalt. Vom Entwurf bis zur Bereinigungssitzung wuchs dieser Haushaltsplan auf knapp 1 Milliarde Euro auf.
Ein Teil dieses Aufwuchses ist nachvollziehbar, da sich einige Zuschüsse und Leistungen an die Herbstprojektion und die Steuerschätzung anlehnen und dadurch zur Bereinigungssitzung angepasst werden. Was mich aber ärgert, sind neue Maßnahmen, die erst zur Bereinigungssitzung auftauchen und für die dann neue Mittel beantragt werden. Im letzten J

Hier wird der Tokenizer von RoBERTa initialisiert.

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-large-finetuned-conll03-german')
dataset = InterruptionDataset(texts, padded_labels, tokenizer, max_len=max_len)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, val_size])


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/886 [00:00<?, ?B/s]

Hier erweitere ich die Klasse LMRobertaForTokenClassification, indem ich wieder Dropout und Gewichtsnormierung einbaue.

In [ ]:
class CustomXLMRobertaForTokenClassification(XLMRobertaForTokenClassification):
    def __init__(self, config):
        super(CustomXLMRobertaForTokenClassification, self).__init__(config)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.utils.weight_norm(nn.Linear(config.hidden_size, config.num_labels))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.roberta(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, position_ids=position_ids, head_mask=head_mask, inputs_embeds=inputs_embeds, output_attentions=output_attentions, output_hidden_states=output_hidden_states, return_dict=return_dict)
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

config = XLMRobertaConfig.from_pretrained('xlm-roberta-large-finetuned-conll03-german')
config.num_labels = 2

model = CustomXLMRobertaForTokenClassification(config)

# Gewichte des vortrainierten Modells laden
pretrained_model = XLMRobertaForTokenClassification.from_pretrained('xlm-roberta-large-finetuned-conll03-german')
model.roberta.load_state_dict(pretrained_model.roberta.state_dict())

model.to(device)

/usr/local/lib/python3.10/dist-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

CustomXLMRobertaForTokenClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, out

Die Abhname des Loss in jeder Epoche sieht bisher vielversprechend aus. Das deutet auf eine Verbesserung der Generalisierungsfähigkeit hin und darauf, dass das Modell die Fehler auf den Trainingsdaten reduziert und lernt. Bei dem Loss der Validierungsdaten fällt von der ersten zur zweiten Epoche auf, dass geringfügig ansteigt. Das könnte auf Overfitting hindeuten. Es ist aber so gering, dass ich vermute, dass das nicht der Fall ist.

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.028400,0.018232
2,0.018500,0.018270
3,0.018400,0.018064


TrainOutput(global_step=3000, training_loss=0.021786638895670573, metrics={'train_runtime': 2651.5833, 'train_samples_per_second': 9.051, 'train_steps_per_second': 1.131, 'total_flos': 2.2288967958528e+16, 'train_loss': 0.021786638895670573, 'epoch': 3.0})

In [ ]:
# Speichern des Modells nach dem Training, sodass es wiederverwendet werden kann, ohne es neu trainieren zu müssen
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/sentencepiece.bpe.model',
 './saved_model/added_tokens.json')

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.018064498901367188, 'eval_runtime': 46.4986, 'eval_samples_per_second': 43.012, 'eval_steps_per_second': 10.753, 'epoch': 3.0}


Leider tritt hier das gleiche Problem wie in 04_BERT (PFAD) auf, da ich nicht die gweichtete Cross-Entropy-Funktion verwende. Die Vorhersagen sind alle für Klasse 0, also "Kein Einwurf".

In [ ]:
model = XLMRobertaForTokenClassification.from_pretrained('./saved_model', num_labels=2).to(device)

test_texts = ["Sehr geehrte Damen und Herren, heute möchte ich über ein wichtiges Thema sprechen. Entschuldigung, darf ich kurz unterbrechen? <interruption> Es geht um die aktuellen wirtschaftlichen Entwicklungen. Die Zahlen zeigen einen positiven Trend, aber wir müssen vorsichtig sein. Ich habe eine Frage zu den genauen Daten, die Sie erwähnt haben. Wir dürfen nicht vergessen, dass viele Menschen noch immer von der Krise betroffen sind. Es ist entscheidend, dass wir weiterhin Maßnahmen zur Unterstützung anbieten. Vielen Dank für Ihre Aufmerksamkeit."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors='pt')
test_encodings = {key: val.to(device) for key, val in test_encodings.items()}  # Daten auf die GPU verschieben
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=-1)

tokens = tokenizer.convert_ids_to_tokens(test_encodings['input_ids'][0])
predictions = predictions[0].cpu().numpy()

formatted_output = ""
for i, (token, prediction) in enumerate(zip(tokens, predictions)):
    if i == 0:
        formatted_output += f"Sentence {i+1}:\n"
    formatted_output += f"{token}: {prediction}\n"
    if token == '</s>':
        formatted_output += "\n"

print(formatted_output)

Some weights of the model checkpoint at ./saved_model were not used when initializing XLMRobertaForTokenClassification: ['classifier.weight_g', 'classifier.weight_v']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at ./saved_model and are newly initialized: ['classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sentence 1:
<s>: 0
▁Sehr: 0
▁ge: 0
ehrt: 0
e: 0
▁Damen: 0
▁und: 0
▁Herren: 0
,: 0
▁heute: 0
▁möchte: 0
▁ich: 0
▁über: 0
▁ein: 0
▁wichtige: 0
s: 0
▁Thema: 0
▁sprechen: 0
.: 0
▁Ent: 0
schuld: 0
igung: 0
,: 0
▁darf: 0
▁ich: 0
▁kurz: 0
▁unter: 0
brechen: 0
?: 0
▁<: 0
inter: 0
rup: 0
tion: 0
>: 0
▁Es: 0
▁geht: 0
▁um: 0
▁die: 0
▁aktuellen: 0
▁: 0
wirtschaftlichen: 0
▁Entwicklung: 0
en: 0
.: 0
▁Die: 0
▁Zahlen: 0
▁zeigen: 0
▁einen: 0
▁positive: 0
n: 0
▁Trend: 0
,: 0
▁aber: 0
▁wir: 0
▁müssen: 0
▁vorsichtig: 0
▁sein: 0
.: 0
▁Ich: 0
▁habe: 0
▁eine: 0
▁Frage: 0
▁zu: 0
▁den: 0
▁genau: 0
en: 0
▁Daten: 0
,: 0
▁die: 0
▁Sie: 0
▁erwähnt: 0
▁haben: 0
.: 0
▁Wir: 0
▁dürfen: 0
▁nicht: 0
▁vergessen: 0
,: 0
▁dass: 0
▁viele: 0
▁Menschen: 0
▁noch: 0
▁immer: 0
▁von: 0
▁der: 0
▁Kris: 0
e: 0
▁betroffen: 0
▁sind: 0
.: 0
▁Es: 0
▁ist: 0
▁entscheiden: 0
d: 0
,: 0
▁dass: 0
▁wir: 0
▁weiterhin: 0
▁Maßnahmen: 0
▁zur: 0
▁Unterstützung: 0
▁anbieten: 0
.: 0
▁Vielen: 0
▁Dank: 0
▁für: 0
▁Ihre: 0
▁Aufmerksamkeit: 0
.: 0
</s>: 0